# 06 · Data Cleaning：数据清洗，脏数据不进向量库

> 加载（04）把文件读进来、解析（05）把结构读出来。这一课在切分之前加一道“闸门”：把确定是噪声、重复、乱码的内容挡在外面。脏文本直接进 chunk，等于把噪声焊进向量，检索和回答会一起变差。

**本文件覆盖知识点**：噪声去除（页眉页脚 / 页码 / 目录 / 水印）/ 控制字符与乱码（OCR 噪声）/ 空白·全半角·Unicode 规整 / 精确去重 / 近重复检测（Jaccard / MinHash）/ 超短·超长·无效内容过滤 / PII 与敏感信息脱敏 / 可复现、可审计的清洗流水线

放进整条链路看它的位置：

```text
doc ──加载(04)──> 解析(05) ──> [06 清洗闸门] ──> 干净文本 ──切分(07)──> chunk
```

> 记住一句话：清洗 = 删掉“确定不是正文”的内容 + 统一格式。删之前问一句“它对检索/回答有没有用”，删完要能数得清删了什么。


In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 为什么清洗：脏进，脏出

- chunk 质量决定检索质量的上限。页眉、页脚、水印会复制进每一块：若“星云智能产品手册 · 第3页”反复出现，向量相似度会被这些模板词带偏，真正的内容反而“看不出差别”。
- 网页转存、OCR、表格提取常带控制字符、零宽空格、乱码（`�`、`﻿`），它们在字符级和子词级上打乱向量分布。
- 同一内容的多份副本（镜像页、重复导出、逐版转写）会形成近重复 chunk，检索 Top-K 被“同一句话的不同版本”占满，等于浪费召回名额。
- 所以清洗是切分（07）、Embedding（10）之前性价比最高的一步：规则一旦写好，每个新文档都自动受益。

> 边界：清洗 ≠ 改写，只做确定性的删除与规整，不要“猜意思去改正文”，以免把内容改错。
> 规则越多越要留日志、可回滚、可对比，这和第 34 课的评测、第 36 课的合规是同一套工程纪律。


In [ ]:
# 先看“脏文本”长什么样：页眉、页码、控制字符、粘连空白、OCR 噪声
zw = chr(0x200b)        # 零宽空格 ZWSP（网页/PDF 复制常见）
bom = chr(0xfeff)       # BOM
ctrl = chr(0) + chr(1)  # 控制字符

DIRTY = (
    '星云智能产品手册    内部资料·请勿外传\n'      # 页眉模板（每个文件重复出现）
    '第 3 页\n'                                    # 页码残留
    '\n'
    '1. 计费方式\n'
    f'基础版价格是{zw}{zw}998 元/月，专业版\n'
    '\n\n\n\n  1999元/月\n'
    f'{ctrl}本产品支持公{zw}有云 SaaS 与私有化部署。\n'
    f'控{bom}制台地址: http://internal.example/console\n'
    '\n'
    '第 4 页\n'
    '2. 常见问题\n'
    '问：如何部署？\n'
    '答：私有化支持容器化部署。\n'
)

print('原始字符数:', len(DIRTY))
print('--- 脏文本（注意不可见字符与空行）---')
print(DIRTY)


## 2. 噪声类型与清洗手段对照

| 噪声 | 典型例子 | 处理手段 | 备注 |
|------|---------|---------|------|
| 页眉 / 页脚 / 水印 | 每页“星云智能产品手册 · 第3页” | 版面坐标删除（来自 05）或行频启发式 | 库内常为固定模板 |
| 目录 / 页码残留 | 独立成行的“第 N 页”、目录行 | 结构规则 / 正则 | PDF 转文本常带 |
| 控制字符 / 零宽 / BOM | `\x00`、ZWSP、`﻿` | 按 Unicode 类别剔除 | OCR、网页复制常见 |
| 多余空白 / 空段 | 连续空格、行首尾空格、`\n\n\n` | 折叠、去行尾、压空行 | 保留段落分隔 |
| 全半角混排 | 全角数字/字母 `ＡＢＣ１２３` | 全半角转换（按需） | 中文标点通常保留全角 |
| 兼容字符 | 上标、`①`、注音 | NFKC（谨慎使用） | 只转确定没歧义的 |
| OCR 粘连 / 乱码 | “空格”被吞、“1”变“l”、“�” | OCR 后处理 + 词表纠错 | 必要时重识别（05） |
| 编码错误 | 锟斤拷、乱码方块 | 源头把 `encoding=` 修对 | 洗不干净就回源头重读 |

> 这些规则高度依赖你自己的文档：先抽样看 20 篇再写规则，并把每条规则和命中数量写进清洗日志。


In [ ]:
# 清洗函数库：纯规则、可解释、可审计（只用标准库）
import re
import unicodedata

# 1) 剔除控制字符 / 零宽字符 / BOM（保留 \t \n \r 这类“格式分隔”）
_KEEP_CTRL = {'\t', '\n', '\r'}

def remove_control_chars(text: str) -> str:
    return ''.join(
        ch for ch in text
        if ch in _KEEP_CTRL or not unicodedata.category(ch).startswith('C')
    )

# 2) 页眉/页脚模板行删除：把“折叠空白后相同”的行视为模板
def _normline(line: str) -> str:
    return re.sub(r'\s+', ' ', line).strip()

_TEMPLATES = {_normline('星云智能产品手册 内部资料·请勿外传')}

def remove_template_lines(text: str) -> str:
    return '\n'.join(ln for ln in text.split('\n') if _normline(ln) not in _TEMPLATES)

# 3) 独立成行的“第 N 页”页码残留
PAGE_LINE = re.compile(r'^第\s*\d+\s*页\s*$')

# 4) 空白规整：全角空格/制表/连续空格 -> 半角空格；行首尾空格去掉；3+ 空行压成 1 空行
def normalize_whitespace(text: str) -> str:
    text = re.sub(r'[　\t ]+', ' ', text)   # 　 = 全角空格
    text = re.sub(r' *\n *', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

# 5) 全半角：全角 英文字母/数字 折成半角，中文标点保留
_CJK_PUNCT = set('，。！？；：“”‘’（）【】、—…·《》')

def to_halfwidth_alnum(text: str) -> str:
    out = []
    for ch in text:
        o = ord(ch)
        if 0xFF01 <= o <= 0xFF5E:               # 全角 ASCII 兼容区
            out.append(ch if ch in _CJK_PUNCT else chr(o - 0xFEE0))
        elif o == 0x3000:                       # 全角空格
            out.append(' ')
        else:
            out.append(ch)
    return ''.join(out)

def clean_basic(text: str) -> str:
    '''顺序：模板行 -> 页码 -> 控制字符 -> 全半角 -> 空白规整'''
    text = remove_template_lines(text)
    text = '\n'.join(ln for ln in text.split('\n') if not PAGE_LINE.match(ln))
    text = remove_control_chars(text)
    text = to_halfwidth_alnum(text)
    text = normalize_whitespace(text)
    return text

# 跑在 c2 的脏样本上
clean = clean_basic(DIRTY)
print('清洗前字符数:', len(DIRTY), '-> 清洗后字符数:', len(clean))
print('--- 清洗结果（页眉/页码/控制字符已消失，空白已规整）---')
print(clean)


## 3. 去重：让同一内容只占一个 chunk

- **精确重复**：规整后的文本完全相同，直接去掉。入库时给每块算 `content_hash`，秒查秒去。
- **近重复**：换行、标点、表述微调导致“长得不完全一样但内容相同”，需要相似度判断。
- 经典做法：把文本切成 n-gram shingle，用 Jaccard（两集合交集/并集）衡量重合度；
  数据量大时改用 MinHash / SimHash：把集合压成固定长度签名，再按签名分桶近似查重。
- 判定阈值要结合你库里的数据调（0.8~0.9 常见）；被删掉的块要记日志，方便回滚和做第 34 课的 A/B。

| 方法 | 粒度 | 代价 | 适用 |
|------|------|------|------|
| 精确 hash | 全文 | 极低 | 镜像页、重复导出 |
| Jaccard（shingle 集合） | 集合 | 中（两两比较） | 小库、人工抽查 |
| MinHash / SimHash | 签名 | 低，可哈希分桶 | 大库、近实时去重 |


In [ ]:
# 近重复检测：shingle + Jaccard（小库演示；大库换 MinHash）
def shingles(text: str, k: int = 3) -> set:
    '''去掉所有空白后取连续 k 字组成集合，抗换行/缩进/空格差异'''
    t = re.sub(r'\s+', '', text)
    return {t[i:i + k] for i in range(len(t) - k + 1)}

def jaccard(a: str, b: str) -> float:
    sa, sb = shingles(a), shingles(b)
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb)

docs = [
    '基础版价格是 998 元/月，专业版是 1999 元/月，企业版请联系销售。',      # 0 原文
    '基础版价格是998元/月，专业版是1999元/月，\n企业版请联系销售。',        # 1 换行/去空格版（近重复）
    'RAG 通过检索外部知识库来增强大模型回答，能有效减少幻觉。',              # 2 不同话题
]
print('相似度矩阵（Jaccard）:')
for i, j in ((0, 1), (0, 2), (1, 2)):
    s = jaccard(docs[i], docs[j])
    tag = '近重复，保留其一' if s > 0.85 else ('相关' if s > 0.5 else '不同话题')
    print(f'  doc{i} vs doc{j}: {s:.2f}  ->  {tag}')

# 精确去重：规整后的全文 hash 即可
import hashlib
def content_hash(text: str) -> str:
    return hashlib.sha1(text.encode('utf-8')).hexdigest()

seen, kept = set(), []
for i, d in enumerate(docs):
    h = content_hash(re.sub(r'\s+', '', d))
    if h in seen:
        print(f'  精确重复，丢弃 doc{i}')
        continue
    seen.add(h); kept.append(d)
print('去重后保留', len(kept), '条。')


In [ ]:
# 知识点·真调说明：近重复检测的盲区 —— “换说法”会让 Jaccard 近乎失灵，模型却能一眼认出是同一信息
# （上面 shingle/Jaccard/hash 是纯算法；这里看它“换说法就漏检”的边界，以及语义判定怎么补位）
import re as _re

def _shingles(text, k=3):
    t = _re.sub(r'\s+', '', text)
    return {t[i:i + k] for i in range(len(t) - k + 1)}

text_a = '本产品支持公有云SaaS与私有化部署；如果智能客服答不上来，会自动转人工，并携带完整对话上下文。'
text_b = '这套系统既能跑在云端托管服务上，也能装进客户自己的机房；当AI助手回答不了客户问题时，会把整段聊天记录交给真人坐席接手。'
sa, sb = _shingles(text_a), _shingles(text_b)
jac = round(len(sa & sb) / len(sa | sb), 2)
print(f'Jaccard(原文, 同义改写) = {jac}  —— 字面重叠极低，纯算法会把它们当成两条不同文本')
_llm_live(
    prompt='入库前抽到两条候选文本，请判定它们是不是“同一信息的换说法”（语义近重复），还是确实不同内容：\n'
           '文本A：本产品支持公有云SaaS与私有化部署；如果智能客服答不上来，会自动转人工，并携带完整对话上下文。\n'
           '文本B：这套系统既能跑在云端托管服务上，也能装进客户自己的机房；当AI助手回答不了客户问题时，'
           '会把整段聊天记录交给真人坐席接手。\n只回答“是/否 + 一句话理由”。',
    system='你是 RAG 数据质检员：只做判定，不修改文本。',
    fallback='是——A、B 的核心信息一致（部署支持公有云/私有化两种方式 + 答不上来自动转人工并带上对话上下文），'
             '只是换了一套说法；若两条都入库，Top-K 会被“同一条信息的不同说法”占满，应做语义去重只留一条。',
    temperature=0.1,
)
print('→ shingle/Jaccard 这类“字面重叠”判定扛不住换说法：上面两段几乎同义，Jaccard 却实测为 0.0。')
print('→ 当库里存在大量同义改写（多版本转写、不同人写的 FAQ）时，可在“规则去重”后再加一道语义层'
      '（Embedding 相似度或 LLM 判定）兜底——这也是本课“先用确定性规则、再按需上更重手段”的分层思路。')

## 4. 过滤与脱敏

- **过滤**：把“没有检索价值”的块提前拿掉。
  - **超短**：空行、`None`、孤立图注（过短的内容既搜不准也答不出）；
  - **超长**：超过切分/模型上限的大块先拦下（上限与你的 chunk_size 有关，见 07 课）；
  - **无效样板**：纯符号、日志行、`TODO`、乱码残留；
  - **无关内容/语言不符**：与知识库语言或主题不符的段落。
- **脱敏（Masking）**：隐私与合规必须在“出库/进库”之前处理，涉及手机号、邮箱、身份证、银行卡、内部域名、账号密码。
  生产做法 = 正则粗筛 + NER 或专用检测服务，替换成占位符而非删掉（保持上下文连贯），并落脱敏日志。

> 脱敏是第 36 课（安全与合规）在数据侧的抓手：清洗阶段是数据进库前最后一个可控点。
> 记得把“含脱敏样本”的问题也放进第 34 课的评测集，确保模型不会照着脱敏前的原文把敏感信息背出来。


In [ ]:
# 过滤与脱敏（阈值只是示例，按你的语料与切分策略定）
MIN_CHARS, MAX_CHARS = 10, 800

def useful_block(text: str) -> bool:
    t = text.strip()
    if not t or len(t) < MIN_CHARS or len(t) > MAX_CHARS:
        return False
    if re.fullmatch(r'[\s\W_]+', t):          # 只剩符号/空白/下划线
        return False
    return True

# 简单 PII 脱敏：替换成占位符，保留原文句式（生产用 NER/专用服务）
# 顺序有讲究：先匹配“更长/更具体”的规则（身份证），否则手机号规则会把身份证咬掉一截。
MASK_RULES = [
    (re.compile(r'\d{17}[\dXx]'), '<身份证>'),                  # 18 位身份证（末位可 X），粗规则仅供演示
    (re.compile(r'1[3-9]\d{9}'), '<手机号>'),
    (re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+'), '<邮箱>'),
]

def mask_pii(text: str) -> str:
    for pat, repl in MASK_RULES:
        text = pat.sub(repl, text)
    return text

sample = '客户电话 13800138000，邮箱 a@b.com 已开通企业版；身份证 110101199001011234 用于备案。'
print('脱敏前:', sample)
print('脱敏后:', mask_pii(sample))


In [ ]:
# 6) 把规则串成“可复现清洗流水线”：固定顺序 + 逐步计数日志
from pathlib import Path

def clean_pipeline(text: str) -> tuple:
    '''固定顺序执行各步；每步记录 清洗前->清洗后 的字符数变化，便于审计与回滚。'''
    steps = []
    def run(name, fn):
        nonlocal text
        before = len(text)
        text = fn(text)
        steps.append((name, before, len(text)))
    def drop_pages(t):
        return '\n'.join(ln for ln in t.split('\n') if not PAGE_LINE.match(ln))

    run('删模板行(页眉/页脚)', remove_template_lines)
    run('删页码残留(第N页)', drop_pages)
    run('剔除控制字符/零宽/BOM', remove_control_chars)
    run('全角折半角', to_halfwidth_alnum)
    run('空白规整', normalize_whitespace)
    run('PII 脱敏', mask_pii)

    paras = [p for p in text.split('\n\n') if useful_block(p)]
    dropped = len(text.split('\n\n')) - len(paras)
    text = '\n\n'.join(paras)
    steps.append(('段落过滤(超短/超长/无效)', -dropped, dropped))  # 用字符差表示“段”

    log = {'清洗前字符': steps[0][1]}
    for name, before, after in steps:
        if name.startswith('段落过滤'):
            log[name] = f'删除 {dropped} 段'
        else:
            log[name] = f'{before} -> {after}'
    log['清洗后字符'] = len(text)
    return text, log

# 真实文档试跑：跑 data/ 下所有 .md（无 data 目录则退回 DIRTY 演示）
corpus = sorted(Path('data').glob('*.md')) if Path('data').is_dir() else []
if corpus:
    for p in corpus:
        raw = p.read_text(encoding='utf-8')
        out, log = clean_pipeline(raw)
        print(f'=== {p.name} ===')
        for k, v in log.items():
            print(f'  {k}: {v}')
        head = out[:60].replace('\n', ' ')
        print('  清洗后开头:', head + '...')
        print()
else:
    out, log = clean_pipeline(DIRTY)
    print('（未找到 data/*.md，用上面的 DIRTY 演示）')
    for k, v in log.items():
        print(f'  {k}: {v}')
    print()
    print('--- 清洗后的正文 ---')
    print(out)


## 小结

- 清洗做三件事：去噪（页眉页脚、控制字符、乱码）、规整（空白、全半角）、去重与过滤（精确/近重复、超短超长）；
- 规则要可解释、可审计、可回滚：每条规则一个函数，逐条记日志；是否真的变好，用第 34 课的评测做 A/B；
- PII 脱敏必须在进库前完成，这是第 36 课安全合规的数据侧要求；
- 清洗只做确定性删除与规整，不“猜意思改写”正文，避免误伤。

清洗后的干净文本，交给下一课切分。07 · Chunking 基础会把它切成语义完整的 chunk。
